In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Import necessary libraries
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('/content/post_feature_engineering_2 (1).csv')

In [4]:
#to make the very tiny outliers which are very common in loan sector to be shown as handles in dataset
#by fact,the outliers are already handled in the feature_engineering_2.ipynb file

def final_cap(df, cols):
    for col in cols:
        #calculate statistical bounds for the CURRENT column
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        #force any value outside these bounds to the limit
        df[col] = np.where(df[col] > upper, upper,
                           np.where(df[col] < lower, lower, df[col]))
    return df

final_cols = ['total_income_log', 'loan_amount_log']
df = final_cap(df, final_cols)

In [36]:
#remove Unnamed_0 its the same as Loan_ID jsut indicating number of rows (Not needed for analysis)
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

#use a dictionary to tell the computer the Level and Units for each column.We made changes to column names to standardise varibale naming cnvetion in phyton
column_info = {
    'gender':            {'Level': 'Nominal', 'Units': '-'},
    'married':           {'Level': 'Nominal', 'Units': '-'},
    'dependents':        {'Level': 'Ordinal', 'Units': 'Count'},
    'education':         {'Level': 'Ordinal', 'Units': '-'},
    'self_employed':     {'Level': 'Nominal', 'Units': '-'},
    'loan_amount_term':  {'Level': 'Discrete',   'Units': 'Months'},
    'credit_history':    {'Level': 'Nominal', 'Units': 'Binary (1/0)'},
    'property_area':     {'Level': 'Nominal', 'Units': '-'},
    'total_income_log':  {'Level': 'Continuous',   'Units': 'log value'},
    'loan_amount_log':   {'Level': 'Continuous',   'Units': 'log value'},
    'loan_status':       {'Level': 'Nominal', 'Units': '-'}
}

In [37]:
table_rows = []

for column in df.columns:
    # grab the info for the current column
    info = column_info.get(column)

    measure_level = info['Level']
    units = info['Units']
    unique_count = df[column].nunique()
    null_count = df[column].isnull().sum()

    # Determine Data Type Display String
    dtype = df[column].dtype
    if 'int' in str(dtype):
        formatted_dtype = "Integer"
    elif 'float' in str(dtype):
        formatted_dtype = "Float"
    else:
        formatted_dtype = "String (Text)"

    #NUMERICAL (RATIO)
    if pd.api.types.is_numeric_dtype(df[column]) and (measure_level == 'Discrete' or measure_level == 'Continuous'):
        type_of_data = "Numerical"

        min_val = round(df[column].min(), 2)
        max_val = round(df[column].max(), 2)
        range_str = f"{min_val} - {max_val}"
        top_val = max_val

        # Calculate Outliers (IQR Method)
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1

        lower_limit = Q1 - 1.5 * IQR
        upper_limit = Q3 + 1.5 * IQR

        # Count outliers
        outlier_condition = (df[column] < lower_limit) | (df[column] > upper_limit)
        outlier_count = outlier_condition.sum()

        # We manually force this to 0 because 360 months is standard and others are valid terms
        if column == 'loan_amount_term':
            outlier_count = 0

        if outlier_count > 0:
            outliers_str = f"Yes ({outlier_count})"
        else:
            outliers_str = "No"

    #CATEGORICAL(NOMINAL/ORDINAL)
    else:
        type_of_data = "Categorical"
        min_val = "-"
        outliers_str = "No" #Categorical data technically doesn't have outliers in this context

        # For encoded columns, show the unique codes
        unique_vals = df[column].dropna().unique()
        unique_vals = sorted(unique_vals) # Sort for better display
        unique_vals_str = [str(x) for x in unique_vals]

        if len(unique_vals_str) > 5:
            range_str = ", ".join(unique_vals_str[:5]) + ", ..."
        else:
            range_str = ", ".join(unique_vals_str)

        #top value is the Mode
        if len(df[column].mode()) > 0:
            top_val = df[column].mode()[0]
        else:
            top_val = "N/A"

    # Append to list
    table_rows.append({
        "Variable": column,
        "Type of Data": type_of_data,
        "Data Type": formatted_dtype,
        "Measurement Level": measure_level,
        "Units": units,
        "Range": range_str,
        "Min Value": min_val,
        "Top Value": top_val,
        "Unique Values": unique_count,
        "Null Values": null_count,
        "Outliers": outliers_str
    })

In [38]:
df.describe()

,dependents,education,self_employed,loan_amount_term,credit_history,property_area,loan_status,total_income_log,loan_amount_log
count,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000
mean,0.827362,0.218241,0.133550,342.410423,0.855049,1.037459,0.687296,8.615365,4.844922
std,1.212833,0.413389,0.340446,64.428629,0.352339,0.787482,0.463973,0.426333,0.409146
min,0.000000,0.000000,0.000000,12.000000,0.000000,0.000000,0.000000,7.462322,3.862506
25%,0.000000,0.000000,0.000000,360.000000,1.000000,0.000000,0.000000,8.334712,4.607658
50%,0.000000,0.000000,0.000000,360.000000,1.000000,1.000000,1.000000,8.597205,4.852030
75%,1.000000,0.000000,0.000000,360.000000,1.000000,2.000000,1.000000,8.916305,5.104426
max,4.000000,1.000000,1.000000,480.000000,1.000000,2.000000,1.000000,9.674978,5.566434


In [39]:
table1_after_preprocessing_df = pd.DataFrame(table_rows)

In [40]:
table1_after_preprocessing_df
#this line shows that the null values and outliers has been handled

,Variable,Type of Data,Data Type,Measurement Level,Units,Range,Min Value,Top Value,Unique Values,Null Values,Outliers
0,dependents,Categorical,Integer,Ordinal,Count,"0, 1, 2, 4",-,0.00,4,0,No
1,education,Categorical,Integer,Ordinal,-,"0, 1",-,0.00,2,0,No
2,self_employed,Categorical,Integer,Nominal,-,"0, 1",-,0.00,2,0,No
3,loan_amount_term,Numerical,Float,Discrete,Months,12.0 - 480.0,12.0,480.00,10,0,No
4,credit_history,Categorical,Float,Nominal,Binary (1/0),"0.0, 1.0",-,1.00,2,0,No
5,property_area,Categorical,Integer,Nominal,-,"0, 1, 2",-,1.00,3,0,No
6,loan_status,Categorical,Integer,Nominal,-,"0, 1",-,1.00,2,0,No
7,total_income_log,Numerical,Float,Continuous,log value,7.46 - 9.67,7.46,9.67,518,0,No
8,loan_amount_log,Numerical,Float,Continuous,log value,3.86 - 5.57,3.86,5.57,156,0,No


In [43]:
table1_after_preprocessing_df.to_csv('table1_after_preprocessing.csv', index=False)